In [1]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import scipy
from scipy.special import kv, gamma
import scipy.linalg
from gaussian_crossings import simulate_gaussian_process_fft, simulate_gaussian_process_cholesky

In [2]:
# 1) Example correlation function r(t)
def r_squared_exp(t, tau, sigma):
    return sigma**2 * torch.exp(- (t / tau)**2)

# 2) Helper function to compute r(t), p(t)=r'(t), q(t)=-r''(t) via autodiff
def compute_r_p_q(t, tau, sigma):
    """
    Returns:
      r_val = r(t)
      p_val = r'(t)
      q_val = -r''(t)
    by using PyTorch's automatic differentiation.
    """
    # Make sure t has requires_grad=True so we can differentiate w.r.t. t
    t = t.clone().detach().requires_grad_(True)
    
    # Compute r(t)
    r_val = r_squared_exp(t, tau, sigma)
    
    # First derivative wrt t: p(t) = r'(t)
    p_val = torch.autograd.grad(r_val, t, create_graph=True)[0]
    
    # Second derivative wrt t: r''(t), then we define q(t) = -r''(t)
    r_second = torch.autograd.grad(p_val, t, create_graph=True)[0]
    q_val = - r_second
    
    return r_val, p_val, q_val

# 3) Example usage:
# Define parameters as tensors (so that if needed, you could also differentiate wrt these)
tau   = torch.tensor(1.0, requires_grad=True)
sigma = torch.tensor(2.0, requires_grad=True)

# Define two time points: t and t0=0
t_value  = torch.tensor(0.1, requires_grad=True)
t0_value = torch.tensor(0.0, requires_grad=True)

# Compute r, p, q at t_value
r, p, q = compute_r_p_q(t_value, tau, sigma)

# Compute r0, p0, q0 at t=0
r0, p0, q0 = compute_r_p_q(t0_value, tau, sigma)

# Now define alpha, beta, gamma, delta
# (You can treat 'u' as a given constant or tensor.)
u = torch.tensor(1.0)

# Use sqrt(2) as a differentiable Tensor
sqrt2 = torch.sqrt(torch.tensor(2.0))

alpha = - (r + r0) / (2 * (p**2 + (q - q0)*(r + r0)))
beta  = - (r0 - r) / (2 * (p**2 + (q + q0)*(r - r0)))
gamma = (sqrt2 * p / (r + r0)) * u
delta = 1.0 / (r + r0)

# Print them just to see something
print("r(t)  =", r.item())
print("p(t)  =", p.item())
print("q(t)  =", q.item())
print("r0    =", r0.item())
print("p0    =", p0.item())
print("q0    =", q0.item())

print("alpha =", alpha.item())
print("beta  =", beta.item())
print("gamma =", gamma.item())
print("delta =", delta.item())


r(t)  = 3.9601993560791016
p(t)  = -0.7920398712158203
q(t)  = 7.761990547180176
r0    = 4.0
p0    = -0.0
q0    = 8.0
alpha = 3.140674114227295
beta  = 1941.1163330078125
gamma = -0.14071425795555115
delta = 0.12562499940395355
